# Analysis Template: Inhibition
Updated 9/17/25, Nicholas Freitas

In [ ]:
#enables autoreloding of modules
%load_ext autoreload
%autoreload 2

from mercury.db_api.mercury_db_api import LocalMercuryDBAPI
from mercury.analysis.experiment import MercuryExperiment

#enable inline plotting of matplotlib figures
%matplotlib inline

#set the figure format to SVG
%config InlineBackend.figure_format = 'svg'

## 1. Connect DB Api

In [ ]:
### PARAMETERS:
EGFP_SLOPE = 91900.03
EGFP_SLOPE_CONC_UNITS = 'nM' #RFU/nM

root = './ic50_data/'
db_conn = LocalMercuryDBAPI(
    standard_curve_data_path= root + 'd3_2_StandardSeries_Analysis.csv',
    standard_name="standard_5-FAM", 
    standard_substrate="5-FAM", 
    standard_units="uM",
    kinetic_data_path= root+ 'd3_TitrationSeries_Analysis.csv',
    kinetic_name="kinetics_AVI4516", 
    kinetic_substrate="NSP 4/5", 
    kinetic_units="nM")

mercury_experiment = MercuryExperiment(db_conn)

## 2. Enzyme Quant

In [ ]:
from mercury.analysis.transform import transform_data

button_concentrations = transform_data(
    data_objs = [mercury_experiment.get_run('button_quant')],              # e.g. a Data2D or Data3D or Data4D instance
    expr=f'(a_luminance / {EGFP_SLOPE})',             # e.g. "(luminance - intercept) / slope"
    output_name='concentration'       # name of the new field, e.g. "concentration"
)

mercury_experiment.set_run('enzyme_concentrations', button_concentrations)     # save the fit results

In [ ]:
mercury_experiment.plot_enzyme_concentration_chip(analysis_name='enzyme_concentrations', units=EGFP_SLOPE_CONC_UNITS)

## 2. Product Standards

In [ ]:
from mercury.analysis.fit import fit_luminance_vs_concentration

standard_experiment_data = mercury_experiment.get_run('standard_5-FAM')       # retrieve our raw data
standard_fits = fit_luminance_vs_concentration(standard_experiment_data)    # perform a fit
mercury_experiment.set_run('standard_5-FAM_fits', standard_fits)              # save the fit results

In [ ]:
mercury_experiment.plot_standard_curve_chip('standard_5-FAM_fits', 'standard_5-FAM')

## 3. Fit Initial Rates

In [ ]:
# Calculate product concentrations from RFU data:
product_concentrations = transform_data(
    data_objs = [mercury_experiment.get_run('kinetics_AVI4516'), mercury_experiment.get_run('standard_5-FAM_fits')],
    expr=f'(a_luminance - b_intercept) / b_slope',             # e.g. "(luminance - intercept) / slope"
    output_name='concentration'       # name of the new field, e.g. "concentration"
)

mercury_experiment.set_run('kinetics_AVI4516_conc', product_concentrations)

In [ ]:
from mercury.analysis.fit import fit_concentration_vs_time

kinetics_concentrations = mercury_experiment.get_run('kinetics_AVI4516_conc')       # retrieve our raw data
kinetics_fits = fit_concentration_vs_time(kinetics_concentrations, start_timepoint = 1, end_timepoint=6)      # perform a fit, skipping the first timepoint
mercury_experiment.set_run('kinetics_AVI4516_conc_fits', kinetics_fits)                # save the fit results

In [ ]:
mercury_experiment.plot_initial_rates_chip(analysis_name='kinetics_AVI4516_conc_fits', experiment_name='kinetics_AVI4516_conc')#, remove_0_point=True) # plot the initial rates

## 4. Filter initial rates

In [ ]:
# These are the "Masks" we use to select which data we want.

from mercury.analysis.filter import filter_expression_cutoff, filter_initial_rates_positive_cutoff, filter_initial_rates_r2_cutoff, filter_standard_curve_r2_cutoff

kinetics_fits = mercury_experiment.get_run('kinetics_AVI4516_conc_fits')       # retrieve our raw data

# Initial Rates:
initial_rates_r2_mask =         filter_initial_rates_r2_cutoff(kinetics_fits, r2_cutoff=0.8) # 0.9 R2
initial_rates_positive_mask =   filter_initial_rates_positive_cutoff(kinetics_fits)          # positive slope

# Standard Curve:
standard_curve_r2_mask =        filter_standard_curve_r2_cutoff(standard_fits, kinetics_fits, r2_cutoff=0.9) # 0.9 R2

# Expression:
expression_mask =               filter_expression_cutoff(button_concentrations, kinetics_fits, expression_cutoff=1) # 1 nM

# Save the masks to the experiment
mercury_experiment.set_run('initial_rates_r2_mask', initial_rates_r2_mask)
mercury_experiment.set_run('initial_rates_positive_mask', initial_rates_positive_mask)
mercury_experiment.set_run('standard_curve_r2_mask', standard_curve_r2_mask)
mercury_experiment.set_run('expression_mask', expression_mask)

In [ ]:
# Plot the masks (Do one at a time)
#mercury_experiment.plot_mask_chip(mask_name='initial_rates_r2_mask')
mercury_experiment.plot_mask_chip(mask_name='initial_rates_positive_mask')
#mercury_experiment.plot_mask_chip(mask_name='standard_curve_r2_mask')
#mercury_experiment.plot_mask_chip(mask_name='expression_mask')

In [ ]:
# Apply the masks to the fits and save the results
mercury_experiment.apply_mask(run_name='kinetics_AVI4516_conc_fits', 
                            dep_variables = ['slope', 'intercept'], 
                            save_as = 'kinetics_AVI4516_conc_fits_masked',
                            mask_names = ['initial_rates_r2_mask', 'initial_rates_positive_mask', 'standard_curve_r2_mask', 'expression_mask'])


In [ ]:
mercury_experiment.plot_initial_rates_chip(analysis_name='kinetics_AVI4516_conc_fits_masked', experiment_name='kinetics_AVI4516_conc')#, remove_0_point=True) # plot the initial rates

In [ ]:
mercury_experiment.plot_initial_rates_vs_concentration_chip(analysis_name='kinetics_AVI4516_conc_fits_masked', x_log=True) # plot the initial rates

## 5. Fit Inhibition Constant:

In [ ]:
# I've written MM here, but it will be IC50
from mercury.analysis.fit import fit_initial_rates_vs_concentration_with_function, mm_model, inhibition_model

kinetics_fits_masked = mercury_experiment.get_run('kinetics_AVI4516_conc_fits_masked')       # retrieve our raw data
IC50_fits, IC50_pred_data = fit_initial_rates_vs_concentration_with_function(data = kinetics_fits_masked,
                                model_func = inhibition_model)
mercury_experiment.set_run('kinetics_AVI4516_IC50_fits', IC50_fits)                # save the fit results
mercury_experiment.set_run('kinetics_AVI4516_IC50_pred_data', IC50_pred_data)    # save the predicted data

In [ ]:
from mercury.analysis.filter import filter_r2_cutoff
# Filter out crappy R2 values: 
IC50_fits = mercury_experiment.get_run('kinetics_AVI4516_IC50_fits')
IC50_fits_mask = filter_r2_cutoff(IC50_fits, r2_cutoff=0.75)  # 0.8 R2

mercury_experiment.set_run('IC50_R2_mask', IC50_fits_mask)  # save the fit results

mercury_experiment.plot_mask_chip(mask_name='IC50_R2_mask')

In [ ]:
# Apply the mask to the fits and save the results:
mercury_experiment.apply_mask(run_name='kinetics_AVI4516_IC50_fits', 
                            dep_variables = ['r_min', 'r_max', 'ic50', 'r_squared'], 
                            save_as = 'kinetics_AVI4516_IC50_fits_masked',
                            mask_names = ['IC50_R2_mask'])

In [ ]:
mercury_experiment.plot_ic50_chip(analysis_name='kinetics_AVI4516_conc_fits_masked', 
                                model_fit_name='kinetics_AVI4516_IC50_fits_masked',
                                model_pred_data_name='kinetics_AVI4516_IC50_pred_data',
                                x_log=True) # plot the initial rates

## 6. Export to CSV

In [ ]:
mercury_experiment.export_run_data_raw(run_name='kinetics_AVI4516_IC50_fits_masked')
mercury_experiment.export_run_data_processed(run_name='kinetics_AVI4516_IC50_fits_masked')